# Silver Layer — Electronic Medication Administration Record (`emar`)

**Source:** `<source_schema>.emar` (default `cdr_mimiciv.bronze.emar`)
**Target:** `<target_schema>.emar` (default `cdr_mimiciv.silver.emar`)

**Objective:** Standardize electronic medication administration records while preserving patient and admission identifiers, administration identifiers, medication information, event timing and event status.

**Grain:** One source eMAR record (MIMIC-IV hosp `emar`: one medication administration event recorded at the bedside; `emar_id` = `subject_id-emar_seq`; links to `emar_detail` on `emar_id`, to `pharmacy`/`prescriptions` on `pharmacy_id` and to `poe` on `poe_id`). `subject_id`, `emar_id` and `emar_seq` are documented NOT NULL. Uniqueness of `emar_id` is not established by this notebook's profiling, so it is **profiled only (distinct count reported), not asserted**. Repeated administration identifiers or other apparent duplicates will be investigated, not automatically removed.

**Silver rules:** Preserve every source record and all source columns in their Bronze types (pass-through). Retain medication names, event descriptions, statuses, and source metadata without changing their meanings.

**Data quality:** Validate identifiers, missing values, exact duplicates, and Bronze–Silver row counts. Document source anomalies without automatically correcting or removing records.

**Publication:** The table is published with `CREATE OR REPLACE` (Delta) into `<target_schema>`; downstream Gold depends on it. Nothing is dropped. Hardcoded profiled counts are asserted only when widget `enforce_profiled_counts` is `true`.

**Source-data quality findings:** 1,417,390 eMAR records have a missing
`hadm_id`, and 23,249 records have a missing `scheduletime`.
These values are preserved as NULL without imputation or record removal.
The Bronze and Silver row counts and schemas match, and no exact
duplicate groups were identified.

**Cost note:** large table (tens of millions of rows). Validation uses single-pass aggregates (no `exceptAll`, no repeated full scans).

**Dependencies:** none (reads Bronze only; no joins to other Silver tables).

In [0]:
# Import PySpark functions
from pyspark.sql import functions as F

In [0]:
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

SOURCE_TABLE = f"{BRONZE}.emar"
TARGET_TABLE = f"{SILVER}.emar"

print("Source:", SOURCE_TABLE)
print("Target:", TARGET_TABLE)
print("Enforce profiled counts:", ENFORCE_PROFILED_COUNTS)

Source: cdr_mimiciv.bronze.emar
Target: cdr_mimiciv.silver.emar
Enforce profiled counts: True


In [0]:
# Load eMAR records without filtering or modifying data
bronze_df = spark.table(SOURCE_TABLE)

bronze_df.printSchema()
display(bronze_df.limit(20))

root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- emar_id: string (nullable = true)
 |-- emar_seq: integer (nullable = true)
 |-- poe_id: string (nullable = true)
 |-- pharmacy_id: integer (nullable = true)
 |-- enter_provider_id: string (nullable = true)
 |-- charttime: timestamp (nullable = true)
 |-- medication: string (nullable = true)
 |-- event_txt: string (nullable = true)
 |-- scheduletime: timestamp (nullable = true)
 |-- storetime: timestamp (nullable = true)



subject_id,hadm_id,emar_id,emar_seq,poe_id,pharmacy_id,enter_provider_id,charttime,medication,event_txt,scheduletime,storetime
10000032,22595853,10000032-10,10,10000032-32,86768272,null,2180-05-07T07:51:00.000Z,Heparin,Administered,2180-05-07T08:00:00.000Z,2180-05-07T07:56:00.000Z
10000032,22595853,10000032-11,11,10000032-37,43092008,null,2180-05-07T07:51:00.000Z,Raltegravir,Administered,2180-05-07T08:00:00.000Z,2180-05-07T07:56:00.000Z
10000032,22595853,10000032-12,12,10000032-27,30740602,null,2180-05-07T07:51:00.000Z,Sodium Chloride 0.9% Flush,Flushed,2180-05-07T07:51:00.000Z,2180-05-07T07:56:00.000Z
10000032,22595853,10000032-13,13,10000032-36,null,null,2180-05-07T08:00:00.000Z,Nicotine Patch,Not Applied,2180-05-07T08:00:00.000Z,2180-05-07T07:57:00.000Z
10000032,22595853,10000032-14,14,10000032-35,null,null,2180-05-07T08:00:00.000Z,Furosemide,Not Given,2180-05-07T08:00:00.000Z,2180-05-07T09:27:00.000Z
10000032,22595853,10000032-15,15,10000032-38,null,null,2180-05-07T08:00:00.000Z,Spironolactone,Not Given,2180-05-07T08:00:00.000Z,2180-05-07T09:27:00.000Z
10000032,22595853,10000032-16,16,10000032-42,null,null,2180-05-07T12:00:00.000Z,Ipratropium Bromide Neb,Not Given,2180-05-07T12:00:00.000Z,2180-05-07T11:39:00.000Z
10000032,22595853,10000032-17,17,10000032-33,73243129,null,2180-05-07T11:39:00.000Z,Albuterol Inhaler,Administered,2180-05-07T11:39:00.000Z,2180-05-07T11:40:00.000Z
10000032,22595853,10000032-18,18,10000032-55,12775705,null,2180-05-07T11:54:00.000Z,Furosemide,Administered,2180-05-07T10:00:00.000Z,2180-05-07T11:54:00.000Z
10000032,22595853,10000032-19,19,10000032-56,46308055,null,2180-05-07T11:54:00.000Z,Spironolactone,Administered,2180-05-07T10:00:00.000Z,2180-05-07T11:54:00.000Z


In [0]:
# Inspect the actual Bronze schema before choosing Silver conversions
schema_df = spark.createDataFrame(
    [(field.name, field.dataType.simpleString(), field.nullable)
     for field in bronze_df.schema.fields],
    ["column_name", "bronze_type", "nullable"]
)

display(schema_df)

column_name,bronze_type,nullable
subject_id,int,true
hadm_id,int,true
emar_id,string,true
emar_seq,int,true
poe_id,string,true
pharmacy_id,int,true
enter_provider_id,string,true
charttime,timestamp,true
medication,string,true
event_txt,string,true


In [0]:
# Preserve all eMAR records, columns, and existing data types
silver_df = bronze_df

silver_df.printSchema()

# Ordered expected schema (pass-through of Bronze)
EXPECTED_SCHEMA = {name: dtype for name, dtype in silver_df.dtypes}

root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- emar_id: string (nullable = true)
 |-- emar_seq: integer (nullable = true)
 |-- poe_id: string (nullable = true)
 |-- pharmacy_id: integer (nullable = true)
 |-- enter_provider_id: string (nullable = true)
 |-- charttime: timestamp (nullable = true)
 |-- medication: string (nullable = true)
 |-- event_txt: string (nullable = true)
 |-- scheduletime: timestamp (nullable = true)
 |-- storetime: timestamp (nullable = true)



In [0]:
# Single pass: row counts, missingness for every column, key checks, timestamp order
bronze_count = bronze_df.count()

critical_columns = [c for c in ["subject_id", "emar_id", "emar_seq"] if c in silver_df.columns]

aggs = [F.count("*").alias("total_records")]
aggs += [
    F.sum(F.col(c).isNull().cast("long")).alias(f"null_{c}")
    for c in silver_df.columns
]
if "emar_id" in silver_df.columns:
    # Profiled only: uniqueness of emar_id is NOT asserted
    aggs.append(F.countDistinct("emar_id").alias("distinct_emar_id"))
if {"charttime", "scheduletime"}.issubset(silver_df.columns):
    aggs.append(F.sum(
        (
            F.col("charttime").isNotNull() &
            F.col("scheduletime").isNotNull() &
            (F.col("charttime") < F.col("scheduletime"))
        ).cast("long")
    ).alias("charttime_before_scheduletime"))
if {"storetime", "charttime"}.issubset(silver_df.columns):
    aggs.append(F.sum(
        (
            F.col("storetime").isNotNull() &
            F.col("charttime").isNotNull() &
            (F.col("storetime") < F.col("charttime"))
        ).cast("long")
    ).alias("storetime_before_charttime"))

profile = silver_df.agg(*aggs).first().asDict()
display(spark.createDataFrame([(k, int(v or 0)) for k, v in profile.items()], ["metric", "value"]))

silver_count = profile["total_records"]

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)

metric,value
total_records,42808593
null_subject_id,0
null_hadm_id,1417390
null_emar_id,0
null_emar_seq,0
null_poe_id,0
null_pharmacy_id,8314698
null_enter_provider_id,36753267
null_charttime,0
null_medication,2120873


Bronze rows: 42808593
Silver rows: 42808593


In [0]:
# Identify exact duplicate records without removing them (single pass)
dup_row = (
    silver_df
    .groupBy(*silver_df.columns)
    .count()
    .filter(F.col("count") > 1)
    .agg(
        F.count("*").alias("duplicate_groups"),
        F.coalesce(F.sum("count"), F.lit(0)).alias("records_in_duplicate_groups"),
        F.coalesce(F.max("count"), F.lit(0)).alias("maximum_group_size")
    )
    .first()
)
duplicate_count = dup_row["duplicate_groups"]

print("Exact duplicate eMAR groups:", duplicate_count)
print("Records in duplicate groups:", dup_row["records_in_duplicate_groups"])

Exact duplicate eMAR groups: 0
Records in duplicate groups: 0


In [0]:
# Pre-publish validation
critical_nulls = {f"null_{c}": profile[f"null_{c}"] for c in critical_columns}
print("Critical NULL counts:", critical_nulls)

assert bronze_count == silver_count, "Bronze–Silver row-count mismatch"
assert silver_df.schema == bronze_df.schema, "Source schema changed"
assert all(v == 0 for v in critical_nulls.values()), (
    f"Documented NOT NULL identifiers contain NULLs: {critical_nulls}"
)

if ENFORCE_PROFILED_COUNTS:
    assert profile["null_hadm_id"] == 1417390, "null hadm_id count changed; review before publication"
    assert profile["null_scheduletime"] == 23249, "null scheduletime count changed; review before publication"
    assert duplicate_count == 0, "Exact duplicate eMAR groups require review"

print("All pre-publish checks passed.")

Critical NULL counts: {'null_subject_id': 0, 'null_emar_id': 0, 'null_emar_seq': 0}
All pre-publish checks passed.


In [0]:
%sql
drop table cdr_mimiciv.silver.emar;

In [0]:
# Publish all eMAR records (CREATE OR REPLACE)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

silver_df.writeTo(TARGET_TABLE).using("delta").createOrReplace()

print("Published:", TARGET_TABLE)

Published: cdr_mimiciv.silver.emar


In [0]:
# Verify published eMAR row count, schema and identifiers (single aggregate pass)
published_df = spark.table(TARGET_TABLE)

published_schema = {name: dtype for name, dtype in published_df.dtypes}
assert list(published_schema.items()) == list(EXPECTED_SCHEMA.items()), (
    f"Published schema differs from expected: {published_schema}"
)

published_stats = published_df.agg(
    F.count("*").alias("rows"),
    *[F.sum(F.col(c).isNull().cast("long")).alias(f"null_{c}") for c in critical_columns]
).first().asDict()

print("Bronze rows:", bronze_count)
print("Published rows:", published_stats["rows"])

assert published_stats["rows"] == bronze_count, "Published row-count mismatch"
assert all(v == 0 for k, v in published_stats.items() if k.startswith("null_")), (
    "Published identifiers contain NULLs"
)

print("Silver eMAR table verified successfully.")

Bronze rows: 42808593
Published rows: 42808593
Silver eMAR table verified successfully.


In [0]:
dbutils.notebook.exit("emar:OK")